# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

In [ ]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1
months = np.unique(month)
month_mask = month == months[:, None]
monthly_revenue = (amount * (month_mask & clean)).sum(axis=1)
september, october = monthly_revenue
change = october / september - 1

print(f"Чистых заказов: {clean.sum()}")
print(f"Сентябрь: {september:,.0f} ₸ | Октябрь: {october:,.0f} ₸ | Изменение: {change:+.1%}")
print("В ДЗ 2 (пункт 5) было 57 489 850 ₸ и 44 419 510 ₸ (−22,7%).")
print("После исключения дублей и TEST оценка падения выручки отличается от ДЗ 2.")

## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2
categories = np.unique(cat)
category_mask = cat == categories[:, None]
category_month_mask = category_mask[:, None, :] & month_mask & clean
category_revenue = (amount * category_month_mask).sum(axis=2)
category_total = category_revenue.sum(axis=1)
category_change = category_revenue[:, 1] / category_revenue[:, 0] - 1

print(f"{'Категория':<12} {'Сентябрь, ₸':>14} {'Октябрь, ₸':>14} {'Изменение':>11}")
for name, values, rate in zip(categories, category_revenue, category_change):
    print(f"{name:<12} {values[0]:>14,.0f} {values[1]:>14,.0f} {rate:>+11.1%}")

top = category_total.argmax()
worst = category_change.argmin()
print(f"Лидер за два месяца — {categories[top]} ({category_total[top]:,.0f} ₸), "
      f"сильнее всего упала категория «{categories[worst]}» ({category_change[worst]:+.1%}).")

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3
category_counts = category_month_mask.sum(axis=2)
average_check = np.divide(
    category_revenue, category_counts,
    out=np.full_like(category_revenue, np.nan), where=category_counts > 0,
)
october_check = average_check[:, 1]

print(f"{'Категория':<12} {'Заказы в сен.':>13} {'Заказы в окт.':>13} {'Чек в окт., ₸':>14}")
for name, counts, check in zip(categories, category_counts, october_check):
    print(f"{name:<12} {counts[0]:>13} {counts[1]:>13} {check:>14,.2f}")

top = np.nanargmax(october_check)
print(f"Самый высокий средний чек в октябре у категории «{categories[top]}» — {october_check[top]:,.2f} ₸.")

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4
cities = np.unique(city)
city_mask = city == cities[:, None]
city_month_mask = city_mask[:, None, :] & month_mask & clean
table = (amount * city_month_mask).sum(axis=2)
city_share = table / table.sum(axis=0)
column_sums = city_share.sum(axis=0)
assert np.allclose(column_sums, 1)

print(f"{'Город':<12} {'Сентябрь':>9} {'Октябрь':>9}")
for name, shares in zip(cities, city_share):
    print(f"{name:<12} {shares[0]:>9.2%} {shares[1]:>9.2%}")
print("Суммы столбцов:", np.round(column_sums, 10))

shymkent_share = city_share[cities == "Шымкент"][0]
share_change = (shymkent_share[1] - shymkent_share[0]) * 100
print(f"Доля Шымкента в общей выручке изменилась с {shymkent_share[0]:.2%} "
      f"до {shymkent_share[1]:.2%} ({share_change:+.2f} п. п.).")

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5
city_change = table[:, 1] - table[:, 0]
total_change = city_change.sum()
contribution = city_change / total_change

print(f"{'Город':<12} {'Изменение, ₸':>14} {'Вклад':>9}")
for name, delta, share in zip(cities, city_change, contribution):
    print(f"{name:<12} {delta:>+14,.0f} {share:>+9.2%}")

shymkent_contribution = contribution[cities == "Шымкент"][0]
print(f"Начните проверку с Шымкента: на него приходится {shymkent_contribution:.2%} общего падения выручки.")

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6
# table имеет форму (6, 2), а суммы по строкам — (6,).
# Broadcasting сравнивает оси справа: размеры 2 и 6 несовместимы.
# keepdims=True сохраняет форму (6, 1), чтобы делить каждую строку на её сумму.
share = table / table.sum(axis=1, keepdims=True)
assert np.allclose(share.sum(axis=1), 1)

print(f"{'Город':<12} {'Сентябрь':>9} {'Октябрь':>9}")
for name, shares in zip(cities, share):
    print(f"{name:<12} {shares[0]:>9.2%} {shares[1]:>9.2%}")
print("В каждом городе доли сентября и октября в выручке за два месяца в сумме дают 100%.")

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8